In [ ]:
import torch
import torch.nn as nn
from tqdm import tqdm

from red import UNet3D
from qsmloader import QSMLoader
from torch.utils.data import DataLoader
from utils import plot_3d_medical_image
import matplotlib.pyplot as plt
from loss import QSM_Loss
from scipy import io
from utils import continuous_dipole_kernel
import numpy as np

In [ ]:
torch.cuda.init()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
model = UNet3D(in_channels=1, out_channels=3)
model = model.to(device)
model.load_state_dict(torch.load('./saved/ph_checkpoint_epoch_5_respaldo.pth', weights_only=False)['model'])
model.eval()

In [ ]:
def eval_model(model_in, p, mask):
    model_in.eval()
    with torch.inference_mode():
        out = model_in((p).to(device)).cpu() * mask
    
    output = out * 1
    lam = .01
    output[:, 0, :, :, :] = torch.minimum(output[:, 0, :, :, :], output[:, 1, :, :, :] )
    output[:, 2, :, :, :] = torch.maximum(output[:, 2, :, :, :], output[:, 1, :, :, :] )
    upper_edge = lam * (output[:, 2, :, :, :] - output[:, 1, :, :, :]) + output[:, 1, :, :, :]
    prediction = output[:, 1, :, :, :]
    lower_edge = output[:, 1, :, :, :] - lam * (output[:, 1, :, :, :] - output[:, 0, :, :, :])
    
    upper_edge = upper_edge[0].numpy()
    prediction = prediction[0].numpy()
    lower_edge = lower_edge[0].numpy()
    return upper_edge, prediction, lower_edge

In [ ]:
ds_calib = QSMLoader(list(range(500)), root='./calib/')
calib_dl = DataLoader(ds_calib, batch_size=10, shuffle=True)

In [ ]:
loss = QSM_Loss()

In [ ]:
preds = []
gts = []
masks = []
for _ in range(1):
    for phase, gt, mask, phase_sr in tqdm(calib_dl):
        model.eval()
        mask = mask.to(device)
        with torch.inference_mode():
            out = model((phase).to(device)) * mask
            out[:, 0, ...] = loss.get_phase(out[:, 0, ...])
            out[:, 1, ...] = loss.get_phase(out[:, 1, ...])
            out[:, 2, ...] = loss.get_phase(out[:, 2, ...])
        out = out.cpu()
        mask = mask.cpu()
        preds.append(out*mask)
        gts.append(gt*mask)
        masks.append(mask)


In [ ]:
model = model.cpu()
torch.cuda.empty_cache()

In [ ]:
preds = torch.cat(preds, dim=0)
gts = torch.cat(gts, dim=0)
masks = torch.cat(masks, dim=0)
preds.shape, gts.shape, masks.shape

In [ ]:
def gen_bounds(output, lam):
    output[:, 0, :, :, :] = torch.minimum(output[:, 0, :, :, :], output[:, 1, :, :, :] )
    output[:, 2, :, :, :] = torch.maximum(output[:, 2, :, :, :], output[:, 1, :, :, :] )
    upper_edge = lam * (output[:, 2:3, :, :, :] - output[:, 1:2, :, :, :]) + output[:, 1:2, :, :, :]
    # prediction = output[:, 1, :, :, :]
    lower_edge = output[:, 1:2, :, :, :] - lam * (output[:, 1:2, :, :, :] - output[:, 0:1, :, :, :])
    return lower_edge, upper_edge

In [ ]:
lower_edge, upper_edge = gen_bounds(preds, .5)
lower_edge.shape, upper_edge.shape, gts.shape, masks.shape

In [ ]:
def get_outside_factor(lower_edge, upper_edge, ground_truth, mask):
    factor = (ground_truth[mask]>=lower_edge[mask]) & (ground_truth[mask]<=upper_edge[mask])
    return factor.float().mean()

In [ ]:
lower_edge, upper_edge = gen_bounds(preds, 1)
get_outside_factor(lower_edge, upper_edge, gts, masks==1)

In [ ]:
lambdas = np.linspace(1.075, 1.1, 5)[:1]
factors = []

for lamb in tqdm(lambdas):
    lower_edge, upper_edge = gen_bounds(preds, lamb)
    f = get_outside_factor(lower_edge, upper_edge, gts, masks==1)
    factors.append(f)


In [ ]:
factors, lambdas

In [ ]:
plot_3d_medical_image(masks[0,0])

In [ ]:
pset = preds[0]
label = gts[0]

In [ ]:
sses = (pset[:, 0].squeeze() > label.squeeze()).float() + (pset[2].squeeze() < label.squeeze()).float()

In [ ]:
from scipy.stats import binom
from scipy.optimize import brentq

def h1(y, mu):
    return y*np.log(y/mu) + (1-y)*np.log((1-y)/(1-mu))

### Log tail inequalities of mean
def hoeffding_plus(mu, x, n):
    return -n * h1(np.minimum(mu,x),mu)

def bentkus_plus(mu, x, n):
    return np.log(max(binom.cdf(np.floor(n*x),n,mu),1e-10))+1

### UCB of mean via Hoeffding-Bentkus hybridization
def HB_mu_plus(muhat, n, delta, maxiters=1000):
    def _tailprob(mu):
        hoeffding_mu = hoeffding_plus(mu, muhat, n)
        bentkus_mu = bentkus_plus(mu, muhat, n)
        return min(hoeffding_mu, bentkus_mu) - np.log(delta)
    if _tailprob(1-1e-10) > 0:
        return 1
    else:
        try:
            print('jiji')   
            return brentq(_tailprob, muhat, 1-1e-10, maxiter=maxiters)
        except:
            print(f"BRENTQ RUNTIME ERROR at muhat={muhat}")
            return 1.0

In [ ]:
HB_mu_plus(0.9, 5000, 0.1)